In [2]:
# CELL 1: Dataset loading and preprocessing

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from sklearn.svm import SVC
from sklearn.preprocessing import MinMaxScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score,
                              classification_report, confusion_matrix)


df = pd.read_csv('email_phishing_dataset_FINAL.csv')

df = df.dropna(subset=['email_from_domain'])
df['domain_age_missing'] = df['domain_age'].isnull().astype(int)
df['domain_age']         = df['domain_age'].fillna(0)

# ENCODING NON-NUMERICAL FEATURES

df_encoded = df.copy()

# email_from_domain → frequency encoding
domain_freq = df_encoded['email_from_domain'].value_counts(normalize=True)
df_encoded['email_domain_freq'] = df_encoded['email_from_domain'].map(domain_freq)
df_encoded['email_domain_freq'] = df_encoded['email_domain_freq'].fillna(0)
df_encoded.drop(['email_from_domain'], axis=1, inplace=True)

# web_geo_loc → frequency encoding
geo_freq = df_encoded['web_geo_loc'].value_counts(normalize=True)
df_encoded['geo_freq'] = df_encoded['web_geo_loc'].map(geo_freq).fillna(0)
df_encoded.drop(['web_geo_loc'], axis=1, inplace=True)

# web_tld → trust score
trusted_tlds = ['com', 'org', 'net', 'edu', 'gov']
df_encoded['tld_trust_score'] = df_encoded['web_tld'].apply(
    lambda x: 1.0 if x in trusted_tlds else 0.0
)
df_encoded.drop(['web_tld'], axis=1, inplace=True)

# web_who_is → binary
df_encoded['web_who_is'] = df_encoded['web_who_is'].map({'complete': 1, 'incomplete': 0})

# web_https → binary
df_encoded['web_https'] = df_encoded['web_https'].map({'yes': 1, 'no': 0})

# Drop noisy features
df_encoded.drop(['web_url', 'web_ip_add', 'web_content'], axis=1, inplace=True)

print("Encoding complete")
print("Final shape:", df_encoded.shape)

X = df_encoded.drop('final_label', axis=1)
y = df_encoded['final_label']

X_train, X_test_df, y_train, y_test_df = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = MinMaxScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=X_train.columns
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test_df), columns=X_test_df.columns
)

mi_scores = mutual_info_classif(X_train_scaled, y_train, random_state=42)
mi_df = pd.DataFrame({
    'feature':  X_train.columns,
    'mi_score': mi_scores
}).sort_values('mi_score', ascending=False)

top_features = mi_df.head(6)['feature'].tolist()
X_train_selected = X_train_scaled[top_features]
X_test_selected  = X_test_scaled[top_features]

print('Dataset loaded and preprocessed.')
print(f'Full training pool: {len(X_train_selected)} samples')
print(f'Full test set:      {len(X_test_selected)} samples')
print(f'Top 6 features: {top_features}')

Encoding complete
Final shape: (7822, 37)
Dataset loaded and preprocessed.
Full training pool: 6257 samples
Full test set:      1565 samples
Top 6 features: ['web_js_len', 'js_obfuscation_ratio', 'web_js_obf_len', 'content_num_scripts', 'content_entropy', 'domain_trust_score']


In [3]:
# CELL 2

N_TRAIN_IBM = 22  
N_TEST_IBM  = 12

np.random.seed(42)

y_train_arr = y_train.values
y_test_arr  = y_test_df.values

# Stratified sample 
idx_train0 = np.where(y_train_arr == 0)[0]
idx_train1 = np.where(y_train_arr == 1)[0]

n0_train = N_TRAIN_IBM // 2
n1_train = N_TRAIN_IBM - n0_train
idx_train_ibm = np.concatenate([
    np.random.choice(idx_train0, n0_train, replace=False),
    np.random.choice(idx_train1, n1_train, replace=False)
])
np.random.shuffle(idx_train_ibm)

# Stratified sample from the GLOBAL test set 
idx_test0 = np.where(y_test_arr == 0)[0]
idx_test1 = np.where(y_test_arr == 1)[0]

n0_test = N_TEST_IBM // 2
n1_test = N_TEST_IBM - n0_test
idx_test_ibm = np.concatenate([
    np.random.choice(idx_test0, n0_test, replace=False),
    np.random.choice(idx_test1, n1_test, replace=False)
])
np.random.shuffle(idx_test_ibm)

# pi-scaled features for quantum circuit compatibility
X_ibm_train = X_train_selected.values[idx_train_ibm] * np.pi
y_ibm_train = y_train_arr[idx_train_ibm]
X_ibm_test  = X_test_selected.values[idx_test_ibm] * np.pi
y_ibm_test  = y_test_arr[idx_test_ibm]

unique, counts = np.unique(y_ibm_train, return_counts=True)
print(f'  Train: {N_TRAIN_IBM} samples  |  Class distribution: {dict(zip(unique.tolist(), counts.tolist()))}')
print(f'  Test:  {N_TEST_IBM} samples (from global test set)')
print(f'  Balance ratio: {min(counts)/max(counts):.3f}')

n_train_circuits = N_TRAIN_IBM * (N_TRAIN_IBM - 1) // 2
n_test_circuits  = N_TEST_IBM * N_TRAIN_IBM
print(f'  Unique train circuits: {n_train_circuits}')
print(f'  Test circuits:         {n_test_circuits}')
print(f'  Total circuits:        {n_train_circuits + n_test_circuits}')

  Train: 22 samples  |  Class distribution: {0: 11, 1: 11}
  Test:  12 samples (from global test set)
  Balance ratio: 1.000
  Unique train circuits: 231
  Test circuits:         264
  Total circuits:        495


In [4]:
# CELL 3: Build ZZFeatureMap + Ideal local simulation baseline

from qiskit.circuit.library import ZZFeatureMap
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC
import time
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams.update({'font.size': 13})

feature_map = ZZFeatureMap(
    feature_dimension=6,
    reps=1,
    entanglement='linear'
)

print('ZZFeatureMap circuit:')
print(feature_map.decompose().draw(output='text'))
print(f'Circuit depth: {feature_map.decompose().depth()}')
print(f'Gate count:    {feature_map.decompose().count_ops()}')

print('\nSTEP 1: Ideal simulation (local, noise-free)')


t0 = time.time()
qk_sim   = FidelityQuantumKernel(feature_map=feature_map)
qsvm_sim = QSVC(quantum_kernel=qk_sim, C=1.0)
qsvm_sim.fit(X_ibm_train, y_ibm_train)
pred_sim = qsvm_sim.predict(X_ibm_test)

t_sim   = time.time() - t0
acc_sim = accuracy_score(y_ibm_test, pred_sim)
f1_sim  = f1_score(y_ibm_test, pred_sim, average='macro')
cm_sim  = confusion_matrix(y_ibm_test, pred_sim)

print(f'Accuracy:  {acc_sim:.4f}')
print(f'F1 Macro:  {f1_sim:.4f}')
print(f'Time:      {t_sim:.1f}s')
print(f'\nConfusion Matrix (ideal simulation):\n{cm_sim}')
print(classification_report(y_ibm_test, pred_sim, target_names=['Legitimate', 'Phishing']))

sim_kernel_matrix = qk_sim.evaluate(X_ibm_train)
np.save(f'ibm_sim_kernel_{N_TRAIN_IBM}.npy', sim_kernel_matrix)
print(f'Saved: ibm_sim_kernel_{N_TRAIN_IBM}.npy')

ZZFeatureMap circuit:
     ┌───┐┌───────────┐                                             »
q_0: ┤ H ├┤ P(2*x[0]) ├──■──────────────────────────────────■───────»
     ├───┤├───────────┤┌─┴─┐┌────────────────────────────┐┌─┴─┐     »
q_1: ┤ H ├┤ P(2*x[1]) ├┤ X ├┤ P(2*(π - x[0])*(π - x[1])) ├┤ X ├──■──»
     ├───┤├───────────┤└───┘└────────────────────────────┘└───┘┌─┴─┐»
q_2: ┤ H ├┤ P(2*x[2]) ├────────────────────────────────────────┤ X ├»
     ├───┤├───────────┤                                        └───┘»
q_3: ┤ H ├┤ P(2*x[3]) ├─────────────────────────────────────────────»
     ├───┤├───────────┤                                             »
q_4: ┤ H ├┤ P(2*x[4]) ├─────────────────────────────────────────────»
     ├───┤├───────────┤                                             »
q_5: ┤ H ├┤ P(2*x[5]) ├─────────────────────────────────────────────»
     └───┘└───────────┘                                             »
«                                                                   

In [ ]:
# CELL 4: IBM REAL HARDWARE 

IBM_TOKEN = "<REDACTED>"

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
from qiskit_machine_learning.state_fidelities import ComputeUncompute
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from sklearn.svm import SVC

print('Connecting to IBM Quantum')
QiskitRuntimeService.save_account(
    channel='ibm_quantum_platform', token=IBM_TOKEN, overwrite=True
)
service = QiskitRuntimeService(channel='ibm_quantum_platform')
print('Connected.')

print('\nFinding least busy backend')
backend = service.least_busy(operational=True, simulator=False, min_num_qubits=6)
print(f'Selected: {backend.name}  |  Qubits: {backend.num_qubits}')
print(f'Total circuits to run: {n_train_circuits + n_test_circuits}')

print('\nStarting IBM hardware run...')
t_hw_start = time.time()

try:
    # optimization_level=3 that fixes known ComputeUncompute transpilation seam bug that caused generic 'Sampler job failed!' errors
    pm = generate_preset_pass_manager(optimization_level=3, backend=backend)
    sampler_real  = SamplerV2(mode=backend)
    fidelity_real = ComputeUncompute(sampler=sampler_real, pass_manager=pm)

    qk_real = FidelityQuantumKernel(feature_map=feature_map, fidelity=fidelity_real)

   
    print('Generating sample transpiled circuit for visualization...')
    sample_x1 = X_ibm_train[0]
    sample_x2 = X_ibm_train[1]
    circ_x2     = feature_map.assign_parameters(sample_x2)
    circ_x1_inv = feature_map.assign_parameters(sample_x1).inverse()
    from qiskit import QuantumCircuit
    cu_circ = QuantumCircuit(feature_map.num_qubits)
    cu_circ.compose(circ_x2, inplace=True)
    cu_circ.compose(circ_x1_inv, inplace=True)
    cu_circ.measure_all()

    transpiled_circ = pm.run(cu_circ)

    print(f'Original circuit depth:   {cu_circ.decompose().depth()}')
    print(f'Transpiled circuit depth: {transpiled_circ.depth()}')
    print(f'Transpiled gate counts:   {transpiled_circ.count_ops()}')
    print(f'Backend native basis gates: {backend.basis_gates}')

    fig_transpiled = transpiled_circ.draw(
        output='mpl',
        style={'fontsize': 12, 'subfontsize': 9},
        fold=-1,
        scale=1.0,
        idle_wires=False
    )
    fig_transpiled.suptitle(
        f'Transpiled Compute-Uncompute Circuit on {backend.name}\n'
        f'(native basis gates, real qubit layout)',
        fontsize=13, y=1.02
    )
    fig_transpiled.savefig('ibm_transpiled_circuit_22.png', dpi=300,
                          bbox_inches='tight', facecolor='white')
    print('Saved: ibm_transpiled_circuit_22.png')

    # Which physical qubits were actually used 
    if transpiled_circ.layout is not None:
        try:
            initial_layout = transpiled_circ.layout.initial_layout
            print(f'\nQubit mapping (logical -> physical):')
            for i in range(feature_map.num_qubits):
                qubit = initial_layout.get_physical_bits()
            print(initial_layout)
        except Exception:
            pass

    print('Evaluating training kernel matrix on real hardware...')
    matrix_train = qk_real.evaluate(X_ibm_train)

    print('Evaluating test kernel matrix on real hardware...')
    matrix_test = qk_real.evaluate(X_ibm_test, X_ibm_train)

    # Precomputed kernel -> classical SVC completes training instantly
    # (mathematically identical to QSVC.fit() on the same kernel)
    qsvm_real = SVC(kernel='precomputed', C=1.0)
    qsvm_real.fit(matrix_train, y_ibm_train)
    pred_real = qsvm_real.predict(matrix_test)

    t_hw     = time.time() - t_hw_start
    acc_real = accuracy_score(y_ibm_test, pred_real)
    f1_real  = f1_score(y_ibm_test, pred_real, average='macro')
    cm_real  = confusion_matrix(y_ibm_test, pred_real)

    print(f'\nIBM hardware run complete in {t_hw/60:.1f} minutes.')
    print(f'Backend:   {backend.name}')
    print(f'Accuracy:  {acc_real:.4f}')
    print(f'F1 Macro:  {f1_real:.4f}')
    print(f'\nConfusion Matrix (real hardware):\n{cm_real}')
    print(classification_report(y_ibm_test, pred_real, target_names=['Legitimate', 'Phishing']))

    np.save('ibm_hardware_kernel_train_22.npy', matrix_train)
    np.save('ibm_hardware_kernel_test_22.npy',  matrix_test)
    print('Saved: ibm_hardware_kernel_train_22.npy, ibm_hardware_kernel_test_22.npy')

    ibm_results_available = True
    ibm_backend_name = backend.name
    ibm_acc, ibm_f1, ibm_time = acc_real, f1_real, t_hw

except Exception as e:
    print(f'\nIBM run failed: {e}')
    ibm_results_available = False

Connecting to IBM Quantum


qiskit_runtime_service.__init__:WARNING:2026-06-30 17:28:35,362: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().


Connected.

Finding least busy backend


qiskit_runtime_service.backends:WARNING:2026-06-30 17:28:35,981: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-06-30 17:28:40,295: Using instance: open-instance, plan: open


Selected: ibm_fez  |  Qubits: 156
Total circuits to run: 495

Starting IBM hardware run...
Generating sample transpiled circuit for visualization...
Original circuit depth:   35
Transpiled circuit depth: 72
Transpiled gate counts:   OrderedDict({'sx': 45, 'rz': 41, 'cz': 18, 'measure': 6, 'x': 3, 'barrier': 1})
Backend native basis gates: ['cz', 'id', 'rz', 'sx', 'x']
Saved: ibm_transpiled_circuit_22.png

Qubit mapping (logical -> physical):
Layout({
141: <Qubit register=(6, "q"), index=0>,
142: <Qubit register=(6, "q"), index=1>,
143: <Qubit register=(6, "q"), index=2>,
136: <Qubit register=(6, "q"), index=3>,
123: <Qubit register=(6, "q"), index=4>,
124: <Qubit register=(6, "q"), index=5>,
0: <Qubit register=(150, "ancilla"), index=0>,
1: <Qubit register=(150, "ancilla"), index=1>,
2: <Qubit register=(150, "ancilla"), index=2>,
3: <Qubit register=(150, "ancilla"), index=3>,
4: <Qubit register=(150, "ancilla"), index=4>,
5: <Qubit register=(150, "ancilla"), index=5>,
6: <Qubit registe

In [6]:
# CELL 5: Comparison - Ideal Simulation vs Real IBM Hardware 

print(f"{'Configuration':<40} {'Accuracy':>10} {'F1 Macro':>10}")
print('-'*70)
print(f"{'Ideal Local Simulation':<40} {acc_sim:>10.4f} {f1_sim:>10.4f}")
print(f"{'IBM Real Hardware (' + ibm_backend_name + ')':<40} {ibm_acc:>10.4f} {ibm_f1:>10.4f}")

gap = abs(acc_sim - ibm_acc)
print(f'\nSimulation-to-Hardware accuracy gap: {gap:.4f}')

import pandas as pd
comparison_df = pd.DataFrame({
    'Configuration': ['Ideal Simulation', f'IBM Hardware ({ibm_backend_name})'],
    'Accuracy':      [acc_sim, ibm_acc],
    'F1 Macro':      [f1_sim, ibm_f1],
    'N_train':       [N_TRAIN_IBM, N_TRAIN_IBM],
    'N_test':        [N_TEST_IBM, N_TEST_IBM]
})
comparison_df.to_csv('ibm_hardware_comparison_22.csv', index=False)
print('\nSaved: ibm_hardware_comparison_22.csv')


Configuration                              Accuracy   F1 Macro
----------------------------------------------------------------------
Ideal Local Simulation                       0.9167     0.9161
IBM Real Hardware (ibm_fez)                  0.9167     0.9161

Simulation-to-Hardware accuracy gap: 0.0000

Saved: ibm_hardware_comparison_22.csv


In [8]:
sim_k  = np.load(f'ibm_sim_kernel_{N_TRAIN_IBM}.npy')
hw_k   = np.load('ibm_hardware_kernel_train_22.npy')
print("Are kernel matrices identical?", np.allclose(sim_k, hw_k))
print("Max difference in kernel values:", np.max(np.abs(sim_k - hw_k)))

Are kernel matrices identical? False
Max difference in kernel values: 0.0568106285172722


In [9]:
import sys
import qiskit
import qiskit_machine_learning
import sklearn
import numpy as np
import pandas as pd
import platform

print("SOFTWARE VERSIONS")
print(f"Python: {sys.version.split()[0]}")
print(f"Qiskit: {qiskit.__version__}")
print(f"Qiskit Machine Learning: {qiskit_machine_learning.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")


SOFTWARE VERSIONS
Python: 3.14.0
Qiskit: 2.4.2
Qiskit Machine Learning: 0.9.0
Scikit-learn: 1.9.0
NumPy: 2.4.6
Pandas: 3.0.3
